In [ ]:
SPEC={'model': 'Qwen/Qwen3-Embedding-8B', 'dim': 4096, 'slug': 'qwen8b', 'doc_prefix': '', 'release_manifest_sha256': '1e6e2e710e6355d769d8fad7cc57e21c9400d6d029daab91058438d5f3e8f88f', 'baseline_chunk_manifest': {'dataset_version': 'v1', 'dataset_built_at': '2026-10-21T15:56:12+00:00', 'clean_version': 'clean-v1', 'chunk_params': '350/50/40', 'id_scheme': '{text_sha256[:16]}-{order}', 'excluded_ids': 118, 'hash_groups': 1023783, 'group_types': {'single': 1020477, 'same_host_diff_path': 1665, 'url_variant': 1576, 'cross_host': 65}, 'sample_docs_per_part': 0, 'languages': {'en': {'texts': 239775, 'doc_ids': 239918, 'chunks': 250487, 'texts_changed_by_cleaning': 7, 'doc_ids_changed_by_cleaning': 8, 'chunks_not_in_raw': 7, 'chunks_per_text_mean': 1.04, 'chunks_per_text_p50': 1, 'chunks_per_text_p90': 1, 'chunks_per_text_max': 6, 'n_units_p50': 201, 'n_units_p90': 298, 'n_units_max': 435, 'read_seconds': 70.0, 'chunk_seconds': 14.2, 'write_seconds': 2.4, 'chunk_seconds_per_1000_texts': 0.06, 'seconds_per_1000_texts': 0.36}, 'vi': {'texts': 685354, 'doc_ids': 688537, 'chunks': 2128601, 'texts_changed_by_cleaning': 0, 'doc_ids_changed_by_cleaning': 0, 'chunks_not_in_raw': 0, 'chunks_per_text_mean': 3.11, 'chunks_per_text_p50': 3, 'chunks_per_text_p90': 5, 'chunks_per_text_max': 383, 'n_units_p50': 320, 'n_units_p90': 370, 'n_units_max': 438, 'read_seconds': 385.6, 'chunk_seconds': 248.0, 'write_seconds': 32.0, 'chunk_seconds_per_1000_texts': 0.36, 'seconds_per_1000_texts': 0.97}, 'zh': {'texts': 98654, 'doc_ids': 98943, 'chunks': 242465, 'texts_changed_by_cleaning': 34564, 'doc_ids_changed_by_cleaning': 34706, 'chunks_not_in_raw': 68, 'chunks_per_text_mean': 2.46, 'chunks_per_text_p50': 1, 'chunks_per_text_p90': 3, 'chunks_per_text_max': 2454, 'n_units_p50': 256, 'n_units_p90': 352, 'n_units_max': 978, 'read_seconds': 36.7, 'chunk_seconds': 8.0, 'write_seconds': 2.1, 'chunk_seconds_per_1000_texts': 0.08, 'seconds_per_1000_texts': 0.47}}, 'elapsed_seconds': 840.5}, 'baseline_embed_manifest_sha256': ['1813b3ef1ffe7f2e51c30e593051b1914b14b55ab02c4167c4a4c85518123b7c', 'e03b5a8b25da696eebb0746ee145c8db8021b1a157e2d5289ec0120d385be708', '56ef525594b9aa4f734243713b6b86fe5d16e15001f748b62537b38209e0f6a0'], 'expected_old_chunks': 2621553, 'model_revision': '1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', 'model_package_manifest_sha256': '144f48d34f50c2bc86ed271ad23d7c4a5934868c32c85b748a9470dc625dd951'}
SOURCE='"""Kaggle CPU inventory for pinned Qwen8B/Octen8B additions; derived from the verified 4B inventory."""\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport time\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for data in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(data)\n    return h.hexdigest()\n\n\ndef one(pattern):\n    found=list(Path(\'/kaggle/input\').rglob(pattern))\n    assert len(found)==1,(pattern,[str(p) for p in found])\n    return found[0]\n\n\ndef run(spec):\n    started=time.monotonic()\n    model, dim, prefix = spec[\'model\'], spec[\'dim\'], spec[\'doc_prefix\']\n    assert dim == 4096 and {"Qwen/Qwen3-Embedding-8B": "", "Octen/Octen-Embedding-8B": " "}[model] == prefix\n    out=Path(\'/kaggle/working\') / (spec[\'slug\']+\'_additions_plan\');out.mkdir(exist_ok=True)\n    release_path=one(\'release_manifest.json\')\n    assert sha(release_path)==spec[\'release_manifest_sha256\']\n    release=json.loads(release_path.read_text())\n    assert release[\'parent_release_manifest_sha256\'], \'Require finalized corpus\'\n    assert release[\'baseline_manifest\']==spec[\'baseline_chunk_manifest\']\n    source=release_path.parent/\'chunk_corpus_additions_20261029_v2.parquet\'\n    assert sha(source)==release[\'files\'][source.name][\'sha256\']\n    manifests=list(Path(\'/kaggle/input\').rglob(\'embed_manifest.json\'))\n    assert {sha(p) for p in manifests}==set(spec[\'baseline_embed_manifest_sha256\'])\n    for p in manifests:\n        m=json.loads(p.read_text())\n        assert m[\'model\']==model and m[\'dim\']==dim and m[\'doc_prefix\']==prefix\n        assert m[\'dtype\']==\'torch.bfloat16\' and m[\'max_length\']==1024 and m[\'batch\']==16\n        assert m[\'complete\'] and m[\'nan_rows\']==0 and m[\'parts_done\']==m[\'parts_total\']==3\n    rows=pq.read_table(source,columns=[\'chunk_id\',\'chunk_text\',\'lang\',\'text_sha256\']).to_pylist()\n    assert len(rows)==release[\'new_chunks\']\n    new_ids={}\n    wanted=set()\n    for i,r in enumerate(rows):\n        h=hashlib.sha256((prefix+r[\'chunk_text\']).encode()).hexdigest()\n        assert r[\'chunk_id\'] not in new_ids\n        r.update(source_chunk_row=i,input_sha256=h)\n        new_ids[r[\'chunk_id\']]=h;wanted.add(h)\n    old_seen=set();reuse={};already={};parts=[];golden=[];golden_vectors=[];vector_paths={}\n    chunks=sorted(Path(\'/kaggle/input\').rglob(\'chunks/*/part-*.parquet\'))\n    assert len(chunks)==9\n    for path in chunks:\n        key=path.parent.name+\'-\'+path.stem\n        vector_path=one(\'vectors/\'+key+\'.npy\');ids_path=one(\'vectors/\'+key+\'-ids.parquet\')\n        vec=np.load(vector_path,mmap_mode=\'r\',allow_pickle=False)\n        ids=pq.read_table(ids_path,columns=[\'chunk_id\']).column(0).to_pylist()\n        n=pq.ParquetFile(path).metadata.num_rows\n        assert vec.dtype==np.float16 and vec.shape==(n,dim) and len(ids)==n\n        vector_paths[key]=vector_path\n        sample={0,n//2,n-1};offset=0\n        for batch in pq.ParquetFile(path).iter_batches(batch_size=4096,columns=[\'chunk_id\',\'chunk_text\',\'lang\']):\n            current=batch.to_pylist()\n            assert [r[\'chunk_id\'] for r in current]==ids[offset:offset+len(current)]\n            for j,r in enumerate(current):\n                cid=r[\'chunk_id\'];pos=offset+j\n                assert cid not in old_seen,\'Duplicate old vector ID\'\n                old_seen.add(cid)\n                h=hashlib.sha256((prefix+r[\'chunk_text\']).encode()).hexdigest()\n                if h in wanted:reuse.setdefault(h,(key,pos,cid))\n                if cid in new_ids:\n                    assert new_ids[cid]==h,\'Existing ID has changed embedding text\'\n                    already[cid]=(key,pos)\n                if pos in sample:\n                    v=np.array(vec[pos],copy=True)\n                    assert np.isfinite(v).all() and abs(float(np.linalg.norm(v.astype(np.float32)))-1)<0.01\n                    golden.append(dict(chunk_id=cid,chunk_text=r[\'chunk_text\'],lang=r[\'lang\'],old_part=key,old_row=pos))\n                    golden_vectors.append(v)\n            offset+=len(current)\n        assert offset==n\n        parts.append(dict(part=key,rows=n,ids_sha256=sha(ids_path),vectors_sha256=sha(vector_path)))\n        print(f\'Verified {key}: {n:,} aligned rows; reusable new inputs so far {len(reuse):,}\',flush=True)\n        del vec,ids\n    assert len(old_seen)==spec[\'expected_old_chunks\']\n    needed_hashes={r[\'input_sha256\'] for r in rows if r[\'chunk_id\'] not in already}\n    reuse_keys=sorted(set(reuse)&needed_hashes);reuse_index={h:i for i,h in enumerate(reuse_keys)}\n    copied=np.lib.format.open_memmap(out/\'reused_old_vectors.npy\',mode=\'w+\',dtype=np.float16,shape=(len(reuse_keys),dim)) if reuse_keys else np.empty((0,dim),dtype=np.float16)\n    by_part=defaultdict(list)\n    for h in reuse_keys:by_part[reuse[h][0]].append(h)\n    reuse_provenance=[]\n    for key,values in by_part.items():\n        vec=np.load(vector_paths[key],mmap_mode=\'r\',allow_pickle=False)\n        for h in values:\n            _,old_row,cid=reuse[h];idx=reuse_index[h];v=np.array(vec[old_row],copy=True)\n            assert np.isfinite(v).all() and abs(float(np.linalg.norm(v.astype(np.float32)))-1)<0.01\n            copied[idx]=v\n            reuse_provenance.append(dict(reuse_row=idx,input_sha256=h,old_chunk_id=cid,old_part=key,old_row=old_row,vector_sha256=hashlib.sha256(v.tobytes()).hexdigest()))\n        del vec\n    if reuse_keys:copied.flush();del copied\n    else:np.save(out/\'reused_old_vectors.npy\',copied)\n    np.save(out/\'golden_vectors.npy\',np.stack(golden_vectors))\n    pq.write_table(pa.Table.from_pylist(golden),out/\'golden_texts.parquet\',compression=\'zstd\')\n    if reuse_provenance:pq.write_table(pa.Table.from_pylist(reuse_provenance),out/\'reuse_provenance.parquet\',compression=\'zstd\')\n    actions=Counter();canonical={};writer=None;buffer=[];count=0\n    schema=pa.schema([(\'output_row\',pa.int64()),(\'source_chunk_row\',pa.int64()),(\'chunk_id\',pa.string()),\n        (\'input_sha256\',pa.string()),(\'chunk_text\',pa.string()),(\'lang\',pa.string()),(\'action\',pa.string()),\n        (\'old_reuse_row\',pa.int64()),(\'canonical_output_row\',pa.int64())])\n    for r in rows:\n        if r[\'chunk_id\'] in already:continue\n        h=r[\'input_sha256\'];old=-1;prior=-1\n        if h in reuse_index:action=\'copy_old\';old=reuse_index[h]\n        elif h in canonical:action=\'copy_new\';prior=canonical[h]\n        else:action=\'infer\';canonical[h]=count\n        buffer.append(dict(output_row=count,source_chunk_row=r[\'source_chunk_row\'],chunk_id=r[\'chunk_id\'],input_sha256=h,\n            chunk_text=r[\'chunk_text\'],lang=r[\'lang\'],action=action,old_reuse_row=old,canonical_output_row=prior))\n        actions[action]+=1;count+=1\n        if len(buffer)>=4096:\n            if writer is None:writer=pq.ParquetWriter(out/\'embedding_tasks.parquet\',schema,compression=\'zstd\')\n            writer.write_table(pa.Table.from_pylist(buffer,schema=schema));buffer=[]\n    if buffer:\n        if writer is None:writer=pq.ParquetWriter(out/\'embedding_tasks.parquet\',schema,compression=\'zstd\')\n        writer.write_table(pa.Table.from_pylist(buffer,schema=schema))\n    if writer:writer.close()\n    else:pq.write_table(pa.Table.from_pylist([],schema=schema),out/\'embedding_tasks.parquet\')\n    report=dict(model=model,model_revision=spec[\'model_revision\'],dim=dim,doc_prefix=prefix,\n        inference_dtype=\'torch.bfloat16\',output_dtype=\'float16\',max_length=1024,batch=16,\n        instruction=\'\',padding_side=\'left\',pooling=\'last_token\',normalize=True,\n        source_release_manifest_sha256=spec[\'release_manifest_sha256\'],model_package_manifest_sha256=spec[\'model_package_manifest_sha256\'],\n        source_chunks=len(rows),output_chunks=count,already_embedded_chunk_ids=len(already),old_chunks_verified=len(old_seen),\n        actions=dict(actions),old_reuse_unique_inputs=len(reuse_keys),golden_checks=len(golden),old_parts=parts,\n        baseline_modified=False,embeddings_generated=False,elapsed_seconds=round(time.monotonic()-started,1))\n    report[\'files\']={p.name:dict(bytes=p.stat().st_size,sha256=sha(p)) for p in out.iterdir() if p.is_file() and p.name!=\'embedding_plan_manifest.json\'}\n    (out/\'embedding_plan_manifest.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps({k:v for k,v in report.items() if k not in [\'files\',\'old_parts\']},indent=2))\n\n\nif __name__==\'__main__\':run(json.loads(Path(\'/kaggle/working/embedding_spec.json\').read_text()))\n'
from pathlib import Path
import json, subprocess, sys
p=Path('/kaggle/working/prepare_8b_delta.py'); p.write_text(SOURCE, encoding='utf-8')
Path('/kaggle/working/embedding_spec.json').write_text(json.dumps(SPEC))
subprocess.run([sys.executable, '-u', str(p)], check=True)
